# Tiled polygonization tutorial

Polygonize categorical rasters in tiles with `contourrs.shapes_arrow`,
merge same-class neighbors across tile boundaries, and visualize.

We start with a small synthetic example, then apply the same
workflow to a real USDA Cropland Data Layer raster.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from contourrs import shapes_arrow
from matplotlib.colors import ListedColormap
from rasterio.windows import Window

## Synthetic raster

Create a small 128x128 categorical raster with 5 classes and
demonstrate tiled polygonization + merge on it first.

In [ ]:
rng = np.random.default_rng(42)
synthetic = rng.integers(1, 6, size=(128, 128), dtype=np.uint8)
print(f"Shape: {synthetic.shape}, classes: {np.unique(synthetic)}")

### Tiled polygonization (synthetic)

Split into 64x64 tiles, polygonize each, then dissolve to merge
tile-boundary artifacts.

In [ ]:
tile_size = 64
h, w = synthetic.shape
frames = []

for row_off in range(0, h, tile_size):
    for col_off in range(0, w, tile_size):
        tile = synthetic[
            row_off : row_off + tile_size, col_off : col_off + tile_size
        ].copy()
        # Affine: (scale_x, shear_x, offset_x, shear_y, scale_y, offset_y)
        transform = (1.0, 0.0, float(col_off), 0.0, 1.0, float(row_off))
        table = shapes_arrow(tile, connectivity=4, transform=transform)
        if table.num_rows > 0:
            frames.append(gpd.GeoDataFrame.from_arrow(table))

tiled_syn = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), geometry="geometry")
tiled_syn["value"] = tiled_syn["value"].astype("int32")
print(f"Raw tiled polygons: {len(tiled_syn):,}")

merged_syn = tiled_syn.dissolve(by="value", as_index=False).explode(
    index_parts=False, ignore_index=True
)
merged_syn = merged_syn[~merged_syn.geometry.is_empty]
merged_syn["value"] = merged_syn["value"].astype("int32")
merged_syn = merged_syn[["value", "geometry"]]
print(f"Merged polygons:    {len(merged_syn):,}")

### Plot: synthetic raster vs merged polygons

In [ ]:
colors = ["#2d6a4f", "#52b788", "#d4a373", "#e9c46a", "#e76f51"]
cmap = ListedColormap(colors)
h, w = synthetic.shape

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(synthetic, cmap=cmap, interpolation="nearest", extent=(0, w, h, 0))
axes[0].set_title("Synthetic raster (5 classes)")

color_map = dict(zip(range(1, 6), colors, strict=False))
merged_syn["color"] = merged_syn["value"].map(color_map)
merged_syn.plot(ax=axes[1], color=merged_syn["color"], edgecolor="black", linewidth=0.1)
axes[1].set_title(f"Merged polygons ({len(merged_syn):,})")

for ax in axes:
    ax.set_axis_off()
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

## Real raster: USDA Cropland Data Layer

A 512x512 crop of the 2023 CDL for Polk County, Iowa.
31 land-cover classes, 30 m resolution, EPSG:5070.

In [ ]:
raster_path = Path("data/cdl_2023_polk_512.tif")
if not raster_path.exists():
    raster_path = Path("examples/data/cdl_2023_polk_512.tif")

with rasterio.open(raster_path) as src:
    print(f"Size: {src.width}x{src.height}")
    print(f"CRS: {src.crs}")
    print(f"dtype: {src.dtypes[0]}, nodata: {src.nodata}")
    data = src.read(1)
    print(f"Classes: {len(np.unique(data))}")

### Helper: transform tuple

`contourrs` expects a 6-element affine transform tuple `(a, b, c, d, e, f)`.
Rasterio's `Affine` object needs a quick conversion.

In [ ]:
def transform_tuple(t):
    return (t.a, t.b, t.c, t.d, t.e, t.f)

### Polygonize in tiles

Read the raster in 128x128 tiles, polygonize each tile with
`shapes_arrow`, and collect the results into one GeoDataFrame.

In [ ]:
tile_size = 128
connectivity = 4
frames = []

with rasterio.open(raster_path) as src:
    rows = range(0, src.height, tile_size)
    cols = range(0, src.width, tile_size)
    total_tiles = len(rows) * len(cols)
    print(
        f"Raster {src.width}x{src.height}, tile_size={tile_size}, tiles={total_tiles}"
    )

    for row_off in rows:
        for col_off in cols:
            window = Window.from_slices(
                (row_off, min(row_off + tile_size, src.height)),
                (col_off, min(col_off + tile_size, src.width)),
            )
            block = src.read(1, window=window)
            mask = block != 0
            if src.nodata is not None:
                mask &= block != src.nodata
            if not mask.any():
                continue

            table = shapes_arrow(
                block,
                mask=mask,
                connectivity=connectivity,
                transform=transform_tuple(src.window_transform(window)),
            )
            if table.num_rows == 0:
                continue

            tile_gdf = gpd.GeoDataFrame.from_arrow(table)
            tile_gdf = gpd.GeoDataFrame(
                {
                    "value": tile_gdf["value"],
                    "geometry": tile_gdf.geometry,
                },
                geometry="geometry",
                crs=tile_gdf.crs,
            )
            frames.append(tile_gdf)

    crs = src.crs

tiled = gpd.GeoDataFrame(
    pd.concat(frames, ignore_index=True),
    geometry="geometry",
)
tiled["value"] = tiled["value"].astype("int32")
tiled.set_crs(crs, inplace=True)

print(f"Raw tiled polygons: {len(tiled):,}")

### Merge same-class neighbors

Dissolve polygons that share the same class value, then explode
multipolygons back to individual geometries. This merges
tile-boundary artifacts.

In [ ]:
merged = tiled.dissolve(by="value", as_index=False).explode(
    index_parts=False, ignore_index=True
)
merged = merged[~merged.geometry.is_empty]
merged["value"] = merged["value"].astype("int32")
merged = merged[["value", "geometry"]]

print(f"Merged polygons: {len(merged):,}")
print(f"Classes: {sorted(merged['value'].unique())}")

### Plot: CDL raster vs merged polygons

Side-by-side comparison of the original CDL raster and the
merged vector polygons.

In [ ]:
with rasterio.open(raster_path) as src:
    raster = src.read(1)
    bounds = src.bounds

extent = (bounds.left, bounds.right, bounds.bottom, bounds.top)
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(raster, cmap="tab20", interpolation="nearest", extent=extent)
axes[0].set_title("CDL raster")

merged.plot(
    ax=axes[1],
    column="value",
    cmap="tab20",
    edgecolor="black",
    linewidth=0.03,
    legend=False,
)
axes[1].set_title(f"Merged polygons ({len(merged):,})")

for ax in axes:
    ax.set_axis_off()
    ax.set_xlim(bounds.left, bounds.right)
    ax.set_ylim(bounds.bottom, bounds.top)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()